# EVCOMX - dados sobre carga alta

Uma corrida pode ter dois cestos de sucata, os carregamentos 1 e 2. Cada cesto recebe seu próprio evento. O carregamento 3 fica na inspeção dos dados, pois não tem evento específico na tabela de resposta.

In [ ]:
from pathlib import Path
import json
import os
import numpy as np
import pandas as pd

os.environ.setdefault('MPLCONFIGDIR', '/tmp/evcomx-matplotlib')
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import MaxNLocator
import seaborn as sns
from IPython.display import display, Markdown

%matplotlib inline
RAIZ = Path.cwd()
if not (RAIZ / 'data/raw').exists():
    RAIZ = RAIZ.parent
DADOS = RAIZ / 'data/raw'
AZUL, LARANJA, CINZA = '#3979A8', '#DA654B', '#64748B'
sns.set_theme(style='whitegrid', palette=[AZUL, LARANJA], font='DejaVu Sans')
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.titlesize': 11, 'axes.labelsize': 10, 'grid.alpha': .25})
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')


def exibir():
    plt.tight_layout()
    plt.show()


materiais_brutos = pd.read_csv(DADOS / 'infos_materiais.csv', dtype={'Metálico': 'string'}, na_values=['null'])
camadas_brutas = pd.read_csv(DADOS / 'infos_camadas.csv', na_values=['null'],
                            dtype=dict.fromkeys(['cd_corrida', 'cd_codigo_material', 'cd_baia'], 'string'))
resposta = pd.read_csv(DADOS / 'VariavelResposta.csv', dtype={'cd_corrida': 'string'}, na_values=['null'])


## 1. Olhando e preparando os dados

O cadastro contém propriedades dos materiais, as camadas registram os pesos e a posição de cada material e a resposta contém os eventos e seus tempos de parada. Essas tabelas têm níveis diferentes e precisam ser relacionadas antes de comparar características com eventos.

Tamanho das tabelas e valores ausentes:

In [ ]:
fontes = {'Materiais': materiais_brutos, 'Camadas': camadas_brutas, 'Resposta': resposta}
display(pd.DataFrame([{'tabela': nome, 'linhas': len(df), 'colunas': df.shape[1]}
                      for nome, df in fontes.items()]))
ausencias = pd.DataFrame([
    {'tabela': nome, 'variavel': coluna, 'ausentes': df[coluna].isna().sum(),
     'ausentes_pct': df[coluna].isna().mean() * 100}
    for nome, df in fontes.items() for coluna in df.columns])
display(ausencias.loc[ausencias.ausentes > 0])


Energia e rendimento faltam para três insumos não metálicos. Nesses casos, a ausência indica uma propriedade que não se aplica. Baias e horários também têm registros ausentes, mas não tem como inventar onde ou quando houve consumo

A preparação é converter vírgulas decimais, remover somente linhas idênticas e interpretar os horários como UTC. **Pesos são assumidos em kg e rendimento em fração** (talvez precise dar uma confirmada nisso). Valores desconhecidos permanecem ausentes, o rotulo de ausente pra baia eh so na visualização. Pesos zero ficam na inspeção, mas não entram na composição dos cestos ativos.


In [ ]:
materiais = materiais_brutos.copy()
for coluna in ['Energia Elétrica', 'Densidade t/m3', 'Rendimento Metálico %']:
    materiais[coluna] = pd.to_numeric(
        materiais[coluna].astype('string').str.replace(',', '.', regex=False), errors='coerce')
camadas = camadas_brutas.drop_duplicates().copy()
camadas['data_utc'] = pd.to_datetime(camadas['dt_hora_consumo'], utc=True, errors='coerce')
camadas['massa_t'] = camadas['qt_peso_carregamento'] / 1000
assert materiais['Metálico'].is_unique and resposta['cd_corrida'].is_unique
assert camadas['massa_t'].ge(0).all()
for i in [1, 2]:
    flag = resposta[f'is_carga_alta_carregamento_{i}']
    parada = resposta[f'qt_segundos_duracao_parada_carregamento_{i}']
    assert flag.isin([0, 1]).all() and parada.ge(0).all()
    assert flag.eq(parada.gt(0)).all()
assert resposta.is_carga_alta.eq(resposta[[f'is_carga_alta_carregamento_{i}' for i in [1, 2]]].max(axis=1)).all()
assert resposta.qt_segundos_parada_carga_alta.eq(
    resposta[[f'qt_segundos_duracao_parada_carregamento_{i}' for i in [1, 2]]].sum(axis=1)).all()

camadas = camadas.merge(materiais, left_on='cd_codigo_material', right_on='Metálico',
                        how='left', validate='many_to_one')
desconhecidos = sorted(camadas.loc[camadas['Metálico'].isna(), 'cd_codigo_material'].unique())
print(f'Duplicatas exatas removidas: {len(camadas_brutas) - len(camadas)}.')
print(f'Materiais sem cadastro: {", ".join(desconhecidos)}.')
print(f'Registros com peso zero preservados: {camadas.massa_t.eq(0).sum()}.')


## 2. Frequência que ocorre carga alta?

A distribuição dos eventos usa **todas as corridas da resposta**, mesmo quando faltam camadas. O gráfico de parada considera apenas corridas com evento: misturar os muitos zeros esconderia a distribuição do tempo perdido nos casos afetados.


In [ ]:
paradas_positivas = resposta.loc[resposta.is_carga_alta == 1, 'qt_segundos_parada_carga_alta']
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
contagens = resposta.is_carga_alta.value_counts().reindex([0, 1], fill_value=0)
axes[0].bar(['Sem carga alta', 'Com carga alta'], contagens, color=[AZUL, LARANJA])
for x, n in enumerate(contagens):
    axes[0].text(x, n, f'{n:,}\n{n / len(resposta):.1%}', ha='center', va='bottom')
axes[0].set(title='Carga alta por corrida', ylabel='Corridas', ylim=(0, contagens.max() * 1.18))
sns.histplot(paradas_positivas / 60, bins=25, color=LARANJA, ax=axes[1])
axes[1].axvline(paradas_positivas.median() / 60, color=AZUL, linestyle='--', label='Mediana')
axes[1].set(title='Duração das paradas nas corridas afetadas', xlabel='Minutos', ylabel='Corridas')
axes[1].legend()
exibir()
display(Markdown(
    f'**Leitura:** {resposta.is_carga_alta.sum()} de {len(resposta)} corridas '
    f'({resposta.is_carga_alta.mean():.1%}) tiveram carga alta. As paradas acumularam '
    f'**{resposta.qt_segundos_parada_carga_alta.sum() / 3600:.1f} horas nesta amostra**. '
    f'A mediana entre corridas afetadas foi {paradas_positivas.median() / 60:.1f} minutos.'))


## 3. Como são os cestos registrados?

Reunindo cada cesto em uma linha. Calcula massa, volume estimado, densidade efetiva, número de camadas e materiais e proporções de leves e pesados. As classes seguem o [guia EVCOMX](../data/Guia_EVCOMX_EDA.pdf).

O volume é a soma de `massa / densidade do cadastro`: ele não mede compactação ou capacidade real do forno (!!!!). Se faltar densidade de algum material, o volume completo permanece ausente. Cestos sem registros também ficam com características ausentes.


In [ ]:
ativas = camadas.loc[camadas.nu_carregamento.isin([1, 2]) & camadas.massa_t.gt(0)].copy()
chaves = ['cd_corrida', 'nu_carregamento']
leves = ['SHRE', 'ESPS', 'CAVA', 'RETL', 'PACB', 'ESBS', 'MARP', 'MISS']
pesados = ['GUSS', 'GUSL', 'RECA', 'SUGU', 'FOFO', '194032', '118441']
ativas['volume'] = ativas.massa_t / ativas['Densidade t/m3']
ativas['massa_leve'] = ativas.massa_t.where(ativas.cd_codigo_material.isin(leves), 0)
ativas['massa_pesada'] = ativas.massa_t.where(ativas.cd_codigo_material.isin(pesados), 0)
grupo = ativas.groupby(chaves)
caracteristicas = grupo.agg(massa_t=('massa_t', 'sum'), camadas=('nu_camada', 'nunique'),
                           materiais=('cd_codigo_material', 'nunique'))
cobertura_completa = grupo['Densidade t/m3'].count().eq(grupo.size())
caracteristicas['volume_m3'] = grupo['volume'].sum(min_count=1).where(cobertura_completa)
caracteristicas['densidade_t_m3'] = caracteristicas.massa_t / caracteristicas.volume_m3
caracteristicas['leves'] = grupo['massa_leve'].sum() / caracteristicas.massa_t
caracteristicas['pesados'] = grupo['massa_pesada'].sum() / caracteristicas.massa_t
rotulos = pd.concat([
    resposta[['cd_corrida', f'is_carga_alta_carregamento_{i}',
              f'qt_segundos_duracao_parada_carregamento_{i}']].rename(columns={
        f'is_carga_alta_carregamento_{i}': 'carga_alta',
        f'qt_segundos_duracao_parada_carregamento_{i}': 'parada_s'}).assign(nu_carregamento=i)
    for i in [1, 2]], ignore_index=True)
cestos = rotulos.merge(caracteristicas.reset_index(), on=chaves, how='left', validate='one_to_one')
assert len(cestos) == 2 * len(resposta) and not cestos.duplicated(chaves).any()
observados = cestos.dropna(subset=['massa_t']).copy()
medidas = ['massa_t', 'volume_m3', 'densidade_t_m3', 'camadas', 'materiais', 'leves', 'pesados']
nomes = {'massa_t': 'Massa (t)', 'volume_m3': 'Volume estimado (m³)',
         'densidade_t_m3': 'Densidade efetiva (t/m³)', 'camadas': 'Camadas ativas',
         'materiais': 'Materiais distintos', 'leves': 'Proporção de leves', 'pesados': 'Proporção de pesados',
         'nu_carregamento': 'Carregamento', 'carga_alta': 'Carga alta', 'parada_s': 'Parada (s)'}
display(observados.groupby('nu_carregamento')[medidas].median().rename(columns=nomes))


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(13, 6))
for ax, coluna in zip(axes.flat, medidas):
    sns.histplot(data=observados, x=coluna, hue='nu_carregamento', bins=20,
                 palette={1: AZUL, 2: LARANJA}, element='step', ax=ax)
    ax.set(title=nomes[coluna], xlabel='', ylabel='Cestos')
    ax.text(.98, .97, f'n={observados[coluna].notna().sum()}', transform=ax.transAxes,
            ha='right', va='top', fontsize=8)
    legenda = ax.get_legend()
    if legenda:
        legenda.set_title('Cesto')
axes.flat[-1].set_visible(False)
exibir()


Os cestos 1 e 2 têm distribuições diferentes de massa, volume e número de camadas. Por isso, as comparações com carga alta serão feitas separadamente para cada carregamento. As proporções são frações de 0 a 1: por exemplo, 0,25 representa 25% da massa.


### Dados ausentes e valores extremos

Os cestos ausentes permanecem na base com seu evento conhecido. Para comparações físicas, usamos os cestos registrados. Antes de interpretar esse grupo, verificamos se ele é diferente do grupo com registros incompletos.

Para massa e volume, marcamos valores fora de **1,5 × IQR** dentro de cada carregamento. Eles são candidatos a revisão, não erros comprovados. O tratamento é comparar a associação do volume com e sem esses registros; a análise principal preserva todos os valores. Nenhuma densidade é imputada e nenhuma receita é substituída.


In [ ]:
completude = cestos.assign(registrado=cestos.massa_t.notna()).groupby('cd_corrida').registrado.all()
qualidade = resposta.assign(dois_cestos=resposta.cd_corrida.map(completude))
taxas_completude = qualidade.groupby('dois_cestos').is_carga_alta.agg(corridas='size', taxa='mean')
sem_cesto_2 = cestos.loc[cestos.nu_carregamento.eq(2) & cestos.massa_t.isna()]
display(taxas_completude)
print(f'{len(sem_cesto_2)} corridas sem cesto 2 registrado, {sem_cesto_2.carga_alta.sum()} têm evento nesse cesto.')

observados['extremo'] = False
extremos = []
for numero, df in observados.groupby('nu_carregamento'):
    for coluna in ['massa_t', 'volume_m3']:
        q1, q3 = df[coluna].quantile([.25, .75])
        inferior, superior = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
        flag = df[coluna].notna() & ~df[coluna].between(inferior, superior)
        observados.loc[df.index[flag], 'extremo'] = True
        extremos.append({'cesto': numero, 'variavel': nomes[coluna], 'extremos': flag.sum()})
display(pd.DataFrame(extremos))
sensibilidade = pd.DataFrame([
    {'cesto': numero, 'r_com_todos': df.volume_m3.corr(df.carga_alta),
     'r_sem_extremos': df.loc[~df.extremo, 'volume_m3'].corr(df.loc[~df.extremo, 'carga_alta'])}
    for numero, df in observados.groupby('nu_carregamento')])
display(sensibilidade)


## 4. Características associadas à carga alta

A matriz abaixo é **completa para as variáveis numéricas da base de cestos**: características, número do carregamento, evento e parada. Identificadores ficam de fora. A correlação usa os dados conhecidos em cada par, sem preencher ausências.

`r` próximo de zero indica pouca associação **linear**; valores negativos indicam que uma variável tende a diminuir quando a outra aumenta. A matriz reúne os dois carregamentos, mas a tabela seguinte separa os cestos para evitar confundir diferenças de estrutura com diferenças de evento. Cestos de uma mesma corrida não são observações independentes.


In [ ]:
colunas_correlacao = medidas + ['nu_carregamento', 'carga_alta', 'parada_s']
correlacao = observados[colunas_correlacao].corr()
fig, ax = plt.subplots(figsize=(10, 8))
mapa = LinearSegmentedColormap.from_list('evcomx', [LARANJA, 'white', AZUL])
sns.heatmap(correlacao.rename(index=nomes, columns=nomes), annot=True, fmt='.2f',
            cmap=mapa, center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title('Correlação de Pearson — cestos observados')
exibir()

associacoes = pd.DataFrame({numero: df[medidas].corrwith(df.carga_alta)
                           for numero, df in observados.groupby('nu_carregamento')})
associacoes.columns = ['Cesto 1', 'Cesto 2']
display(associacoes.rename(index=nomes))


A correlação entre evento e parada é esperada pela própria definição desses campos, parada é um **desfecho**, não uma variável para prever o evento. Massa, volume e camadas também compartilham informação.

Para responder à pergunta principal, observe as linhas **volume** e **proporção de pesados** na tabela por cesto. Os gráficos abaixo mostram se as distribuições se separam ou se continuam muito sobrepostas. Isso ajuda a interpretar a correlação, sem transformar associação em causa.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for linha, numero in enumerate([1, 2]):
    df = observados.loc[observados.nu_carregamento == numero]
    for ax, coluna in zip(axes[linha], ['volume_m3', 'pesados']):
        sns.boxplot(data=df, x='carga_alta', y=coluna, hue='carga_alta',
                    palette={0: AZUL, 1: LARANJA}, legend=False, fliersize=2, ax=ax)
        ax.set(title=f'Cesto {numero}', xlabel='Carga alta', ylabel=nomes[coluna])
        ax.set_xticks([0, 1], ['Não', 'Sim'])
exibir()


## 5. COMPLEMENTAR: calendário externo

Usamos o [calendário da BrasilAPI](https://brasilapi.com.br/docs), com cópias dos JSONs de [2025](https://brasilapi.com.br/api/feriados/v1/2025) e [2026](https://brasilapi.com.br/api/feriados/v1/2026) em `data/external`. O notebook pode ser executado sem internet.

A referência é o primeiro horário de consumo com peso positivo do cesto 1, convertido para `America/Sao_Paulo`. A comparação conta cada corrida uma vez. O calendário inclui datas como Carnaval e Páscoa, não representa necessariamente dias sem operação da planta. Horários de consumo também não comprovam o início da operação.


In [ ]:
calendario = pd.concat([pd.DataFrame(json.loads(
    (RAIZ / f'data/external/brasilapi_calendario_{ano}.json').read_text()))
    for ano in [2025, 2026]], ignore_index=True)
inicio = ativas.loc[ativas.nu_carregamento == 1].groupby('cd_corrida').data_utc.min()
contexto = resposta[['cd_corrida', 'is_carga_alta']].copy()
contexto['data_local'] = contexto.cd_corrida.map(inicio).dt.tz_convert('America/Sao_Paulo').dt.strftime('%Y-%m-%d')
contexto = contexto.dropna(subset=['data_local'])
assert set(contexto.data_local.str[:4]) <= {'2025', '2026'}
contexto['data_especial'] = contexto.data_local.isin(calendario.date)
taxas_calendario = contexto.groupby('data_especial').is_carga_alta.agg(corridas='size', eventos='sum', taxa='mean')
n, p, z = taxas_calendario.corridas, taxas_calendario.taxa, 1.96
centro = (p + z*z/(2*n)) / (1 + z*z/n)
margem = z * np.sqrt(p*(1-p)/n + z*z/(4*n*n)) / (1 + z*z/n)
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(['Dia comum', 'Data do calendário'], p * 100, color=AZUL)
ax.errorbar([0, 1], p * 100, yerr=np.vstack([p-centro+margem, centro+margem-p]) * 100,
            fmt='none', color=CINZA, capsize=5)
ax.set(title='Calendário e carga alta', ylabel='Taxa de carga alta (%)', ylim=(0, 100))
exibir()
display(taxas_calendario)


Os intervalos de Wilson de 95% mostram a incerteza das taxas, mas não corrigem seleção da amostra ou diferenças de receita. O grupo de datas especiais é pequeno. Uma diferença observada pode refletir dia da semana, período ou composição dos cestos, o gráfico não demonstra um efeito do calendário sobre a operação.


## 6. Sintese

Os números abaixo são calculados a partir desta execução. Eles resumem as constatações da amostra e orientam novas perguntas.


In [ ]:
volume_r = associacoes.loc['volume_m3']
pesados_r = associacoes.loc['pesados']
display(Markdown(
    f'1. **O problema é frequente:** {resposta.is_carga_alta.mean():.1%} das corridas tiveram carga alta, '
    f'com {resposta.qt_segundos_parada_carga_alta.sum() / 3600:.1f} horas acumuladas de parada na amostra.\n\n'
    f'2. **A composição merece investigação:** a proporção de pesados tem associação negativa com o evento '
    f'(r={pesados_r.iloc[0]:+.3f} e {pesados_r.iloc[1]:+.3f}), enquanto a associação linear do volume é pequena '
    f'(r={volume_r.iloc[0]:+.3f} e {volume_r.iloc[1]:+.3f}). Isso não comprova que mudar a receita reduza eventos.\n\n'
    f'3. **As ausências mudam a população observada:** faltam registros do cesto 2 em {len(sem_cesto_2)} corridas. '
    f'A taxa de carga alta em qualquer carregamento é {taxas_completude.loc[False, "taxa"]:.1%} nas corridas '
    f'incompletas e {taxas_completude.loc[True, "taxa"]:.1%} nas completas.\n\n'
    f'4. **O contexto externo traz uma hipótese adicional:** a taxa é {taxas_calendario.loc[True, "taxa"]:.1%} '
    f'nas datas do calendário (n={taxas_calendario.loc[True, "corridas"]}) e '
    f'{taxas_calendario.loc[False, "taxa"]:.1%} nos demais dias. Essa comparação precisa ser contextualizada.'
))


Para apresentar: comece pela frequência e duração das paradas, mostre as diferenças entre cestos, discuta volume e composição e termine com as limitações dos registros. O calendário é um complemento, não a explicação principal.

O próximo passo é esclarecer os materiais sem cadastro e os cestos ausentes e confirmar unidades, fuso e cobertura da amostra com a equipe de processo. A análise ainda não valida uma regra operacional nem um modelo de previsão.
